In [ ]:
import torch
from torch.utils.data import DataLoader
import sentence_transformers as st
from sentence_transformers import losses
import pandas as pd

## Fine tuning a Hugging Face model

In this notebook we fine tune a model from Hugging Face and save it on the local disk.

We perform fine tuning using the triplet learning method where we use 3 inputs for a model:
- Anchor - A reference input
- Positive - An input for which model's output should be similar to the output for the Anchor
- Negative - An input for which model's output should be different than the output for the Anchor

and we train the model in such a way to make the difference between outputs for the Anchor and Positive smaller by at least a specific margin than the difference between outputs for the Anchor and Negative.

To do this, we use the losses.TripletLoss loss function.

In [ ]:
# train_data should have 3 columns: Anchor, Positive and Negative
train_data = pd.read_csv('data/train_data.csv', index_col = 0)

In [122]:
# model = st.SentenceTransformer('sentence-transformers/multi-qa-mpnet-base-dot-v1')
model = st.SentenceTransformer('sebastian-hofstaetter/distilbert-dot-tas_b-b256-msmarco')

In [123]:
train_examples = []
# train_data = pd.read_excel(os.path.join(os.path.dirname(os.getcwd()), 'data', 'anchor_positive_negative_3.xlsx')).values
# train_data = [['anchor1', 'positive1', 'negative1'], ['anchor2', 'positive2', 'negative2']]

for i in range(len(train_data)):
    example = train_data[i]
    train_examples.append(st.InputExample(texts=[example[0], example[1], example[2]]))
    # train_examples.append(st.InputExample(texts=[example[0], example[1]]))

In [ ]:
# Prepare training dataset to be used in the model.fit() method
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

In [125]:
len(train_examples)

148

In [ ]:
# Measures of distance / difference between vectors (will be used to measure distance between model's outputs for Anchor, Positive and Negative)

# Dot product similarity
def dot_sim(x, y):
    return torch.mm(x, y.transpose(0,1))

# Cosine similarity
def cos_sim(a, b):
    norm_a = torch.norm(a, dim = 1)[None, :]
    norm_b = torch.norm(b, dim = 1)[None, :].transpose(0,1)
    return torch.mm(a, b.transpose(0,1)) / (norm_a * norm_b)

# Euclidean distance
def norm_dist(a, b):
    if len(a.shape) == 2:
        # return torch.norm(a - b, dim = 1)[None, :]
        return torch.norm(a - b, dim = 1)[:, None]
    elif len(a.shape) == 1:
        return torch.norm(a - b)

# Use the losses.TripletLoss loss function as it is designed for triplet learning
train_loss = losses.TripletLoss(model = model, distance_metric = norm_dist, triplet_margin = 2)
# train_loss = losses.MultipleNegativesRankingLoss(model = model_trained, scale = 1, similarity_fct = dot_sim)

In [127]:
# evaluator = st.evaluation.EmbeddingSimilarityEvaluator.from_input_examples(train_examples, batch_size=16, name='sts-dev')

model.fit(
    train_objectives = [(train_dataloader, train_loss)],
    epochs = 30,
    show_progress_bar = True
    # evaluation_steps = 1,
    # evaluator = evaluator
)

model.save('models/model_2')

Epoch: 100%|███████████████████████████████████████████████████████████████████████████| 30/30 [28:47<00:00, 57.60s/it]
